# Ders 8: Çok Değişkenli Zaman Serileri
Bu notebook'ta şunları öğreneceğiz:
- Çapraz kovaryans matrisi ve CCF
- VAR(p) modeli: tahmin ve yorumlama
- Gecikme uzunluğu seçimi
- Granger nedensellik testi
- Eşbütünleşme (Engle-Granger)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.stattools import adfuller, coint, grangercausalitytests
from statsmodels.graphics.tsaplots import plot_acf, plot_ccf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
print("Hazır.")


## 8.1 Çok Değişkenli Zaman Serilerinin Özellikleri

**Çapraz kovaryans fonksiyonu (CCVF):**
$$\gamma_{XY}(h) = \text{Cov}(X_{t+h}, Y_t) = E[(X_{t+h}-\mu_X)(Y_t-\mu_Y)]$$

**Çapraz korelasyon fonksiyonu (CCF):**
$$\rho_{XY}(h) = \frac{\gamma_{XY}(h)}{\sqrt{\gamma_X(0)\gamma_Y(0)}}$$

**Dikkat:** $\rho_{XY}(h) \neq \rho_{YX}(h)$ — gecikme yönü önemlidir!
- $\rho_{XY}(h) \neq 0$, $h > 0$: $X$ gecikmeli olarak $Y$'yi etkiliyor


In [ ]:
# ── CCF ile Öncü-Takipçi İlişkisi ──

np.random.seed(42)
n = 300

# X öncü gösterge, Y 3 dönem gecikmeli takip ediyor
eps_x = np.random.normal(0, 1, n)
eps_y = np.random.normal(0, 0.5, n)
X = np.zeros(n)
Y = np.zeros(n)
for t in range(1, n):
    X[t] = 0.7*X[t-1] + eps_x[t]

for t in range(4, n):
    Y[t] = 0.5*X[t-3] + 0.4*Y[t-1] + eps_y[t]  # Y, X'in 3 dönem gerisinde

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0,0].plot(X, label='X (Öncü)', color='steelblue', lw=0.8)
axes[0,0].plot(Y, label='Y (Takipçi)', color='darkorange', lw=0.8)
axes[0,0].set_title('X ve Y Zaman Serileri', fontweight='bold')
axes[0,0].legend()

# CCF
from statsmodels.tsa.stattools import ccf as compute_ccf
lags = np.arange(-20, 21)
cc = [np.corrcoef(X[max(0,-h):n-max(0,h)], Y[max(0,h):n-max(0,-h)])[0,1] for h in lags]
axes[0,1].bar(lags, cc, color='steelblue', alpha=0.7, width=0.8)
axes[0,1].axhline(0, color='black', lw=0.5)
conf = 1.96/np.sqrt(n)
axes[0,1].axhline(conf, color='red', ls='--', alpha=0.7, label='%95 GA')
axes[0,1].axhline(-conf, color='red', ls='--', alpha=0.7)
axes[0,1].set_title('CCF: X ve Y arasında çapraz korelasyon', fontweight='bold')
axes[0,1].set_xlabel('Gecikme h')
axes[0,1].axvline(-3, color='green', ls=':', lw=2, label='h=-3 (beklenen)')
axes[0,1].legend(fontsize=8)

# ACF her ikisi için
plot_acf(X, lags=20, ax=axes[1,0], title='ACF — X')
plot_acf(Y, lags=20, ax=axes[1,1], title='ACF — Y')

plt.tight_layout()
plt.show()
print("h=-3 civarında belirgin CCF: Y, X'in 3 dönem gerisinde izliyor.")


## 8.2 VAR(p) Modeli

**Vektör Otoregresif Model VAR(p):**

$$\mathbf{X}_t = \mathbf{c} + \Phi_1 \mathbf{X}_{t-1} + \Phi_2 \mathbf{X}_{t-2} + \cdots + \Phi_p \mathbf{X}_{t-p} + \mathbf{Z}_t$$

Burada:
- $\mathbf{X}_t = (X_{1t}, X_{2t}, \ldots, X_{kt})^T$: $k$-boyutlu süreç
- $\Phi_i$: $k \times k$ katsayı matrisleri
- $\mathbf{Z}_t \sim WN(\mathbf{0}, \Sigma)$: çok değişkenli gürültü

**Gecikme seçimi:** AIC, BIC, FPE, HQIC kriterleri.


In [ ]:
# ── VAR(2) Simülasyon ve Tahmin ──

np.random.seed(42)
n = 400

# VAR(2) süreci: 2 değişken, 2 gecikme
Phi1 = np.array([[0.5, 0.2],
                  [0.1, 0.6]])
Phi2 = np.array([[-0.1, 0.05],
                  [0.05, -0.15]])
Sigma = np.array([[1.0, 0.5],
                   [0.5, 1.0]])

X = np.zeros((n, 2))
for t in range(2, n):
    noise = np.random.multivariate_normal([0, 0], Sigma)
    X[t] = Phi1 @ X[t-1] + Phi2 @ X[t-2] + noise

df = pd.DataFrame(X, columns=['X1', 'X2'])

fig, axes = plt.subplots(2, 1, figsize=(13, 6))
axes[0].plot(df['X1'], color='steelblue', lw=0.8, label='X1')
axes[1].plot(df['X2'], color='darkorange', lw=0.8, label='X2')
axes[0].set_title('VAR(2) Simüle Veri — X1', fontweight='bold')
axes[1].set_title('VAR(2) Simüle Veri — X2', fontweight='bold')
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.show()

# Model tahmini
model = VAR(df)

# Gecikme seçimi
lag_order = model.select_order(maxlags=8)
print("Gecikme Seçim Kriterleri:")
print(lag_order.summary())


In [ ]:
# ── VAR(2) Katsayı Tahmini ──

var_fit = model.fit(maxlags=2, ic=None)
print(var_fit.summary())


In [ ]:
# ── VAR ile Öngörü ──

forecast_steps = 20
forecast = var_fit.forecast(df.values[-var_fit.k_ar:], steps=forecast_steps)
forecast_df = pd.DataFrame(forecast, columns=['X1_pred', 'X2_pred'])

fig, axes = plt.subplots(2, 1, figsize=(13, 7))

for i, (col, pred_col, color) in enumerate([('X1','X1_pred','steelblue'),
                                              ('X2','X2_pred','darkorange')]):
    axes[i].plot(range(n), df[col], color=color, lw=0.8, alpha=0.6, label='Gözlem')
    pred_idx = range(n, n+forecast_steps)
    axes[i].plot(pred_idx, forecast_df[pred_col], color='red', lw=2, label='Öngörü')
    axes[i].axvline(n, color='black', ls='--', alpha=0.5)
    axes[i].set_title(f'{col} — VAR(2) {forecast_steps} Adım Öngörüsü', fontweight='bold')
    axes[i].legend()

plt.tight_layout()
plt.show()


## 8.3 Granger Nedensellik Testi

**Granger Nedenselliği:** $X$, $Y$'nin Granger nedenidir; eğer $Y$'nin geçmiş değerlerine ek olarak $X$'in geçmiş değerleri de $Y$'yi öngörmeye yardımcı oluyorsa.

$$H_0: X, Y\text{'nin Granger nedeni değildir}$$

**Dikkat:** Bu nedensellik değil, öngörü ilişkisidir!


In [ ]:
# ── Granger Nedensellik Testi ──

print("Granger Nedensellik: X1 -> X2 ?")
print("="*50)
gc_result = grangercausalitytests(df[['X2','X1']], maxlag=4, verbose=True)


## 8.4 Eşbütünleşme (Cointegration)

Eğer iki $I(1)$ (birim köklü) seri $X_t$ ve $Y_t$ durağan olmayan iken, belirli bir lineer kombinasyonları $Y_t - \alpha X_t$ durağan ise, bu seriler **eşbütünleşiktir**.

**Ekonomik örnek:** Fiyat ve maliyet, faiz oranları, döviz paritesi.

**Engle-Granger Testi:**
1. $Y_t = \alpha + \beta X_t + u_t$ regresyonu
2. Artıklar $\hat{u}_t$ üzerinde ADF testi


In [ ]:
# ── Eşbütünleşme Örneği ──

np.random.seed(123)
n = 300

# Ortak stokastik trend (eşbütünleşik çift)
common_trend = np.cumsum(np.random.normal(0, 1, n))
X_coint = common_trend + np.random.normal(0, 0.5, n)
Y_coint = 2 * common_trend + 1 + np.random.normal(0, 0.5, n)

# Bağımsız I(1) seriler (eşbütünleşik değil)
X_indep = np.cumsum(np.random.normal(0, 1, n))
Y_indep = np.cumsum(np.random.normal(0, 1, n))

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0,0].plot(X_coint, label='X', color='steelblue')
axes[0,0].plot(Y_coint, label='Y', color='darkorange')
axes[0,0].set_title('Eşbütünleşik Çift', fontweight='bold')
axes[0,0].legend()

# Artıklar
resid_coint = Y_coint - 2*X_coint - 1
axes[0,1].plot(resid_coint, color='green')
axes[0,1].axhline(0, color='red', ls='--')
axes[0,1].set_title('Artıklar Y - 2X - 1 (Durağan Olmalı)', fontweight='bold')

axes[1,0].plot(X_indep, label='X', color='steelblue')
axes[1,0].plot(Y_indep, label='Y', color='darkorange')
axes[1,0].set_title('Bağımsız I(1) Seriler (Sahte Regresyon)', fontweight='bold')
axes[1,0].legend()

resid_indep = Y_indep - X_indep
axes[1,1].plot(resid_indep, color='red')
axes[1,1].axhline(0, color='black', ls='--')
axes[1,1].set_title('Artıklar Y - X (Durağan Değil)', fontweight='bold')

plt.tight_layout()
plt.show()

# Engle-Granger testi
t1, p1, cv1 = coint(X_coint, Y_coint)
t2, p2, cv2 = coint(X_indep, Y_indep)
print(f"Esbütünlesik çift  — p-degeri: {p1:.4f} {'=> Esbütünlesik ✓' if p1<0.05 else '=> Degil'}")
print(f"Bagimsiz çift      — p-degeri: {p2:.4f} {'=> Esbütünlesik ✓' if p2<0.05 else '=> Degil'}")


## Özet

| Yöntem | Kullanım |
|--------|---------|
| **CCF** | Seriler arası gecikme ilişkisi |
| **VAR(p)** | Çok değişkenli öngörü |
| **Granger** | Öngörü nedenselliği testi |
| **Eşbütünleşme** | Uzun vadeli denge ilişkisi |

**Model Seçimi:**
- Gecikme: AIC/BIC/HQIC
- Durağanlık kontrolü: ADF testi
- Eşbütünleşme varsa: VECM modeli (VAR yerine)
